In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from brevitas.export import export_qonnx
from utils.utils import read_object
from cnn_base import t1_CNN

%matplotlib inline
torch.manual_seed(4)

## 1. Dataset

A classe `SatImgDataset` precisa estar definida antes de desserializar os `.pkl` (mesmo nome/estrutura usada em `0_pickle_dataset.ipynb`).

In [4]:
batch_size = 512
epochs = 30
lr = 3e-4
device = torch.device('cpu')

train_dataset_output_name = "./dataset/train_dataset.pkl"
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_train = read_object(train_dataset_output_name)
loader_train = DataLoader(dataset_train, batch_size, shuffle=True)

dataset_test = read_object(test_dataset_output_name)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

## 2. Treinamento

In [5]:
def train_model(model, loader_train, optimizer, criterion, epochs, device):
    for e in range(epochs):
        model.train()
        running_loss = 0.0
        print(f"epoch ({e})")
        for x,y in tqdm(loader_train):
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long)

            pred = model(x)
            loss = criterion(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        mean_loss = running_loss / len(loader_train)
        print(f"epoch ({e}) loss: {mean_loss:.4f}")

    return model


model = t1_CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

model_output_filename = "./pytorch_models/sat6-cnn-t1.pt"
model = train_model(model, loader_train, optimizer, criterion, epochs, device)

os.makedirs("./pytorch_models", exist_ok=True)
torch.save(model.state_dict(), model_output_filename)
print(f"modelo salvo em {model_output_filename}")

epoch (0)


  0%|          | 0/633 [00:00<?, ?it/s]


TypeError: linear(): argument 'input' (position 1) must be Tensor, not Flatten

## 4. Teste / avaliação

In [ ]:
def test_model(model, loader_test, dataset_test, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x,y in loader_test:
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long)

            pred = model(x)
            predicted = torch.max(pred, 1)[1]
            correct += (predicted == y).sum()
    accuracy = correct.item() / len(dataset_test) * 100
    print(f"acurácia do modelo t1: {accuracy:.4f}% ({len(dataset_test)} casos de teste)")
    return accuracy


test_model(model, loader_test, dataset_test, device)

## 5. Exportação para QONNX (entrada do hls4ml)

Diferente do fluxo com o FINN (que exige as transformações de *streamlining*, *lowering* e conversão para camadas de hardware antes da síntese), o **hls4ml** consegue consumir o modelo diretamente a partir do QONNX exportado pelo Brevitas, aplicando internamente o parsing das camadas quantizadas (`Quant`, `QuantConv`, `QuantLinear`, etc.). Portanto, basta exportar e limpar o grafo ONNX.

In [ ]:
import qonnx.util.cleanup
import onnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.gemm_to_matmul import GemmToMatMul
from qonnx.util.to_channels_last import to_channels_last


os.makedirs("./hls4ml_onnxs", exist_ok=True)
export_onnx_path = "./hls4ml_onnxs/sat6-cnn-t1.onnx"

model.eval()
export_qonnx(model,  torch.randn(1, 4, 28, 28), export_path=export_onnx_path, opset_version=14)
print(f"modelo QONNX exportado em {export_onnx_path}")

# Clean and convert Gemm → MatMul
qonnx.util.cleanup.cleanup(export_onnx_path, out_file=export_onnx_path)
model_wrapper = ModelWrapper(export_onnx_path)
model_wrapper = model_wrapper.transform(GemmToMatMul())
model_wrapper = qonnx.util.cleanup.cleanup_model(model_wrapper)
model_wrapper.save(export_onnx_path)
print(f'Cleaned QONNX saved to {export_onnx_path}')


## Transform in Channel last if it's needed for hls4ml

In [ ]:
to_channels_last(
    in_file=export_onnx_path,
    out_file=export_onnx_path,
    make_input_channels_last=True
)
print(f'Model layout converted to channels_last and saved to {export_onnx_path}')

### Verify if every node has name

If not, create a name for this nodes and save the new onnx model.

In [ ]:
onnx_model = onnx.load(export_onnx_path)

def onnx_node_names(onnx_model):
    graph = onnx_model.graph
    for i, node in enumerate(graph.node):
        print(f"Node {i}: {node.name}")
        if not node.name:
            node.name = f"{node.op_type.lower()}_{i}"
            print(f"Node {i} had no name. Assigned name: {node.name}")
            
    for i, value_info in enumerate(graph.value_info):
        print(f"ValueInfo {i}: {value_info.name}")
        if not value_info.name:
            value_info.name = f"tensor_{i}"
            print(f"ValueInfo {i} had no name. Assigned name: {value_info.name}")

onnx_node_names(onnx_model)   

In [ ]:
# Caso precisar, mas melhor não usar.
def fix_onnx_node_names(model_path, output_path):
    model = onnx.load(model_path)
    graph = model.graph
    
    # 1. Corrige os nós do grafo
    for i, node in enumerate(graph.node):
        if not node.name:  # Se o nome estiver vazio ("")
            # Cria um nome baseado no tipo do operador e no índice
            node.name = f"{node.op_type.lower()}_{i}"
            
    # 2. Garante que os tensores intermediários (valores) também tenham nome se necessário
    for i, value_info in enumerate(graph.value_info):
        if not value_info.name:
            value_info.name = f"tensor_{i}"
            
    onnx.save(model, output_path)
    print(f"Modelo corrigido salvo em: {output_path}")


fix_onnx_node_names(export_onnx_path, export_onnx_path)